# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ SQL Server

In [3]:
df_data_quocgia = pd.read_csv("./data_quoc_gia.csv")
df_data_quocgia = df_data_quocgia.where(pd.notnull(df_data_quocgia), None)
print(df_data_quocgia)

     ID_quoc_gia Ma_ISO        Ten_nuoc_ISO
0              0     NO    (Không xác định)
1              1     AF         Afghanistan
2              2     AL             Albania
3              3     DZ             Algeria
4              4     AS      American Samoa
..           ...    ...                 ...
232          232     AJ          Azerbaijan
233          233     CI             Croatia
234          234     XV            Slovenia
235          235     BN  Bosnia-Hercegovina
236          236     XN           Macedonia

[237 rows x 3 columns]


## Xử lý data

### [Nếu cần] Clear bảng

In [6]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Quoc_gia"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [4]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Quoc_gia (ID_quoc_gia, Ma_ISO, Ten_nuoc_ISO) 
                VALUES (?, ?, ?)
                """
for index, row in df_data_quocgia.iterrows():
    values = (row['ID_quoc_gia'], 
              row['Ma_ISO'],
              row['Ten_nuoc_ISO'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()